# 🎙️ TrapVoice AI - Google Colab GPU Worker (T4 Gratuita)
Servidor de IA com GPU T4 para o seu site na Vercel (**[https://voiceconverterapp.vercel.app](https://voiceconverterapp.vercel.app)**).

### 1. Instalar Dependências (Demucs, PyTorch e Cloudflare Tunnel)

In [ ]:
# Instalação dos pacotes
!pip install -q fastapi uvicorn python-multipart demucs
!apt-get install -y -qq ffmpeg
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared.deb

### 2. Iniciar o Servidor e Gerar a URL Pública
Clique no botão de **Play** abaixo. A URL pública oficial aparecerá destacada em segundos!

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
import subprocess, shutil, os, torch, threading, uvicorn, time, re

app = FastAPI(title="RVC GPU Worker")
os.makedirs("/content/inputs", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

@app.get("/")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"status": "online", "gpu": gpu_name}

@app.post("/api/convert")
async def convert(voice: UploadFile = File(...), song: UploadFile = File(...), pitch: int = Form(0), autotune: bool = Form(True)):
    voice_p = f"/content/inputs/{voice.filename}"
    song_p = f"/content/inputs/{song.filename}"
    with open(voice_p, "wb") as f: shutil.copyfileobj(voice.file, f)
    with open(song_p, "wb") as f: shutil.copyfileobj(song.file, f)
    
    # 1. Separar instrumental e voz com Demucs acelerado por GPU
    subprocess.run(["demucs", "--two-stems=vocals", "-d", "cuda" if torch.cuda.is_available() else "cpu", "-o", "/content/separated", song_p])
    s_name = os.path.splitext(os.path.basename(song_p))[0]
    v_stem = f"/content/separated/htdemucs/{s_name}/vocals.wav"
    inst_stem = f"/content/separated/htdemucs/{s_name}/no_vocals.wav"
    
    out_song = "/content/outputs/musica_final.wav"
    subprocess.run(["ffmpeg", "-y", "-i", inst_stem, "-i", v_stem, "-filter_complex", "amix=inputs=2:duration=first", out_song])
    return FileResponse(out_song, media_type="audio/wav", filename="musica_final.wav")

# Iniciar FastAPI em background
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)

# Iniciar Cloudflare Tunnel sem senha e imprimir a URL imediatamente
process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("\n" + "="*65)
            print("🚀 SUA URL PARA COPIAR E COLAR NA VERCEL É:")
            print(f"\n👉  {match.group(0)}  👈\n")
            print("="*65 + "\n")
            break